In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt

In [132]:

# Генерация данных
n = 1000  # Количество пользователей
np.random.seed(42)  # Фиксируем случайность для воспроизводимости

# Генерация user_id
user_ids = np.random.choice(range(1, n * 10), n, replace=False)  # Уникальные случайные числа

# Генерация loyalty_program (True или False)
loyalty_program = np.random.choice([True, False], size=n)

# Генерация revenue
revenue_A = np.where(
    loyalty_program,
    np.random.normal(loc=3000, scale=np.sqrt(250), size=n),  # Для True: мат. ожидание = 3000, дисперсия = 250
    np.random.normal(loc=2000, scale=np.sqrt(350), size=n)   # Для False: мат. ожидание = 2000, дисперсия = 350
)

# Формирование DataFrame
df = pd.DataFrame({
    'user_id': user_ids,
    'loyalty_program': loyalty_program,
    'revenue_A': revenue_A
})

In [133]:
df.head()

,user_id,loyalty_program,revenue_A
0,5345,False,2025.277365
1,7445,False,2046.558840
2,1732,False,1978.721869
3,8720,True,2966.960994
4,4522,False,2005.918897


In [134]:
# Добавляем влияние нового дизайна для группы B
# Генерация нового столбца (на основе условий)
revenue_B = np.where(
    loyalty_program,
    np.random.normal(loc=3005, scale=np.sqrt(250), size=n),  # Для True: мат. ожидание = 3100, дисперсия = 200
    np.random.normal(loc=2000, scale=np.sqrt(350), size=n)   # Для False: мат. ожидание = 2000, дисперсия = 350
)

# Формирование DataFrame
df['revenue_B'] = revenue_B

In [135]:
df.head()

,user_id,loyalty_program,revenue_A,revenue_B
0,5345,False,2025.277365,2019.593291
1,7445,False,2046.558840,2000.726948
2,1732,False,1978.721869,1985.019733
3,8720,True,2966.960994,2976.859793
4,4522,False,2005.918897,1999.684384


In [149]:
# Разделение данных на группы
group_true = df[df['loyalty_program'] == True]
group_false = df[df['loyalty_program'] == False]

# A/B тест для группы с программой лояльности (True)
t_stat_true, p_value_true = ttest_ind(group_true['revenue_A'], group_true['revenue_B'], equal_var=False)

# A/B тест для группы без программы лояльности (False)
t_stat_false, p_value_false = ttest_ind(group_false['revenue_A'], group_false['revenue_B'], equal_var=False)

# Результаты теста
results = {
    "Loyalty Program (True)": {"t-statistic": t_stat_true, "p-value": p_value_true},
    "No Loyalty Program (False)": {"t-statistic": t_stat_false, "p-value": p_value_false},
}

# Вывод результатов
results_df = pd.DataFrame(results).T
print(results_df)

                            t-statistic   p-value
Loyalty Program (True)        -4.613323  0.000004
No Loyalty Program (False)     0.265260  0.790868


## Без стратификации

In [137]:
# Случайное разбиение пользователей без учета возрастных групп
group_A_random, group_B_random = train_test_split(df, test_size=0.5, random_state=42)

# Добавляем информацию о группе
group_A_random['group'] = 'A'
group_B_random['group'] = 'B'

# Объединяем группы
ab_test_data_random = pd.concat([group_A_random, group_B_random])

In [138]:
# Среднее время для групп A и B
mean_A_random = group_A_random['revenue_A'].mean()
mean_B_random = group_B_random['revenue_B'].mean()

print(f"Среднее (группа A, случайное распределение): {mean_A_random:.4f}")
print(f"Среднее (группа B, случайное распределение): {mean_B_random:.4f}")

Среднее (группа A, случайное распределение): 2514.0439
Среднее (группа B, случайное распределение): 2552.4031


In [139]:
# T-тест без учета стратификации
t_stat_random, p_value_random = ttest_ind(group_A_random['revenue_A'], group_B_random['revenue_B'], equal_var=False)
print(f"\nT-статистика (без стратификации): {t_stat_random:.4f}, p-значение: {p_value_random:.4f}")



T-статистика (без стратификации): -1.2106, p-значение: 0.2263
